# E791 toy generation and signal-only fit

Generate and fit a non-CP $D^+\to\pi^-\pi^+\pi^+$ toy using the E791 Fit-2 isobar model and mass-plane Gauss--Legendre normalization.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, DecayChannel, DecayModel, LASS, Minimizer, NonResonant,
    Parameter, PhaseSpaceSample, RealImag, Resonance, enable_x64,
    weighted_resample,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


## 1. Amplitude model and normalization

In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))

fit2_polar = {
    "sigma": (1.17, 205.7), "rho770": (1.00, 0.0),
    "NR": (0.48, 57.3), "f0_980": (0.43, 165.0),
    "f2_1270": (0.76, 57.3), "f0_1370": (0.26, 105.4),
    "rho1450": (0.14, 319.1),
}

def internal_xy(name):
    magnitude, phase = fit2_polar[name]
    if name == "NR":
        phase += 180.0
    phase = np.deg2rad(phase)
    return magnitude*np.cos(phase), magnitude*np.sin(phase)

truth = {}
def coefficient(name, fixed=False):
    x, y = internal_xy(name)
    if fixed:
        return RealImag(x, y)
    truth[f"{name}.x"], truth[f"{name}.y"] = float(x), float(y)
    return RealImag(
        Parameter.coefficient(f"{name}.x", x, owner=name, step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, step=0.01),
    )

c = {name: coefficient(name, fixed=(name == "rho770")) for name in fit2_polar}
components = [
    Resonance("sigma", (0,1), c["sigma"], mass=0.4780, width=0.3240, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho770", (0,1), c["rho770"], mass=0.7693, width=0.1502, spin=1, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_980", (0,1), c["f0_980"], mass=0.9750, width=0.0440, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f2_1270", (0,1), c["f2_1270"], mass=1.2750, width=0.1850, spin=2, resonance_radius=3.0, parent_radius=3.0),
    Resonance("f0_1370", (0,1), c["f0_1370"], mass=1.4340, width=0.1730, spin=0, resonance_radius=3.0, parent_radius=3.0),
    Resonance("rho1450", (0,1), c["rho1450"], mass=1.4650, width=0.3100, spin=1, resonance_radius=3.0, parent_radius=3.0),
    NonResonant(c["NR"]),
]
model = DecayModel(
    channel, components,
    normalization_method="gauss-legendre",
    normalization_order_m13=180,
    normalization_order_m23=180,
)
norm = model.normalization_sample
print("free parameters:", len(model.parameters))
print("normalization points:", norm.size)
model.print_fit_fractions(truth, normalization_sample=norm, include_interference=True)


## 2. Generate signal pseudo-data

In [ ]:
N_POOL = 200_000
N_DATA = 30_000
pool = model.generate_phase_space(N_POOL, seed=2000)
pool_cache = model.prepare_cache(pool, norm)
target_weights = pool.weights * pool_cache.intensity(truth)
data = weighted_resample(
    jax.random.key(791), pool, target_weights, N_DATA, replace=True
)
print("generated events:", data.size)

fig, ax = plt.subplots(figsize=(7, 5.5))
h = ax.hist2d(np.asarray(data.s12), np.asarray(data.s13), bins=90)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
plt.show()


## 3. Unbinned fit and closure table

In [ ]:
cache = model.prepare_cache(data, norm)
def nll(values):
    intensity, normalization = cache.evaluate(values)
    return -jnp.sum(jnp.log(jnp.clip(intensity, min=1e-300))) + data.size*jnp.log(normalization)

rng = np.random.default_rng(314159)
start = {
    parameter.name: truth[parameter.name] + rng.normal(0.0, 0.15)
    for parameter in model.parameters if not parameter.fixed
}
result = Minimizer(nll, model.parameters, verbose=1).fit(
    start_values=start, simplex=True, ncall=30_000
)
fit_values = {
    parameter.name: float(result.values[parameter.name])
    for parameter in model.parameters if not parameter.fixed
}
print("valid:", result.valid, "NLL:", result.fval, "EDM:", result.fmin.edm)
print(f"{'parameter':18s} {'generated':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for parameter in model.parameters:
    if parameter.fixed:
        continue
    name = parameter.name
    fitted = float(result.values[name]); error = float(result.errors[name])
    pull = (fitted-truth[name])/error
    print(f"{name:18s} {truth[name]:11.5f} {fitted:11.5f} {error:11.5f} {pull:9.3f}")

model.print_fit_fractions(
    fit_values, normalization_sample=norm, include_interference=True
)


## Interpretation

A single pseudoexperiment is a workflow and closure demonstration. Bias and coverage require an ensemble of statistically independent toys.